# Notebook 06 — Compare two Genie spaces

**What you’ll do:** Run the **same** benchmark questions on:

- **Main space** — instructions **and** examples (from **02**), plus benchmarks from **03**.
- **No-example space** — **same** instructions **without** in-space examples.

**Why it matters:** Side-by-side pass rates show how much **examples and benchmarks** improve answers.

**Before you start:** **01**, **02**, **03**.


## Compute

Use **Serverless**. Catalog and schema come from **widgets**.


## Configuration

Reads both space IDs from `workshop_config` (created in **02**).


In [ ]:
dbutils.widgets.text("catalog", "gm_ama_demos", "Catalog")
dbutils.widgets.text("schema", "genie_workshop_manufacturing", "Schema")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
fqn = f"{CATALOG}.{SCHEMA}"

from databricks.sdk import WorkspaceClient
import requests, time, re
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from datetime import datetime

w = WorkspaceClient()
host = w.config.host.rstrip("/")
auth = {**w.config.authenticate(), "Content-Type": "application/json"}


def _cfg(key):
    rows = spark.sql(
        f"SELECT value FROM {fqn}.workshop_config WHERE key = '{key}'"
    ).collect()
    return rows[0]["value"] if rows else None


SID_WITH = _cfg("genie_space_id")
SID_NO = _cfg("genie_space_id_configured_no_evals")
if not SID_WITH:
    raise RuntimeError("Missing genie_space_id — run notebook 02.")
if not SID_NO:
    raise RuntimeError("Missing genie_space_id_configured_no_evals — run notebook 02.")
print("WITH examples:", SID_WITH)
print("WITHOUT examples:", SID_NO)

import html


def genie_ui_room_url(host: str, space_id: str) -> str:
    h = (host or "").rstrip("/")
    sid = str(space_id or "").strip()
    m = re.search(r"adb-(\d+)\.", h)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{h}/genie/rooms/{sid}{q}"


for _label, _sid in (
    ("WITH in-space examples", SID_WITH),
    ("WITHOUT in-space examples (instructions only)", SID_NO),
):
    _u = genie_ui_room_url(host, _sid)
    print(_label, "Genie UI URL:", _u)
    displayHTML(
        '<p><b>'
        + html.escape(_label)
        + '</b> &mdash; <a href="'
        + html.escape(_u, quote=True)
        + '" target="_blank" rel="noopener">Open in Genie</a></p>'
    )


def ask_genie(question, space_id, timeout=90):
    try:
        s = requests.post(
            f"{host}/api/2.0/genie/spaces/{space_id}/start-conversation",
            headers=auth,
            json={"content": question},
        )
        if s.status_code != 200:
            return None, f"HTTP {s.status_code}"
        d = s.json()
        cid, mid = d.get("conversation_id"), d.get("message_id")
        for _ in range(timeout // 3):
            time.sleep(3)
            p = requests.get(
                f"{host}/api/2.0/genie/spaces/{space_id}/conversations/{cid}/messages/{mid}",
                headers=auth,
            )
            if p.status_code != 200:
                continue
            msg = p.json()
            if msg.get("status") == "COMPLETED":
                for att in msg.get("attachments", []):
                    if att.get("query"):
                        aid = att.get("attachment_id") or att.get("id", "")
                        qr = requests.get(
                            f"{host}/api/2.0/genie/spaces/{space_id}/conversations/{cid}/messages/{mid}/query-result/{aid}",
                            headers=auth,
                        )
                        if qr.status_code != 200:
                            qr = requests.get(
                                f"{host}/api/2.0/genie/spaces/{space_id}/conversations/{cid}/messages/{mid}/query-result",
                                headers=auth,
                            )
                        if qr.status_code == 200:
                            arr = (
                                qr.json()
                                .get("statement_response", {})
                                .get("result", {})
                                .get("data_array", [])
                            )
                            if arr and arr[0]:
                                m = re.search(
                                    r"-?\d+\.?\d*",
                                    str(arr[0][0]).replace(",", "").replace("$", ""),
                                )
                                if m:
                                    return round(float(m.group())), None
                return None, "No numeric value"
            if msg.get("status") in ("FAILED", "CANCELLED"):
                return None, msg.get("status")
        return None, "Timeout"
    except Exception as e:
        return None, str(e)[:80]


def run_benchmarks(benchmarks_list, space_id, label="Run"):
    results = []
    print(label)
    for i, b in enumerate(benchmarks_list, 1):
        try:
            gt = round(float(spark.sql(b["gt"]).collect()[0][0]))
        except Exception:
            gt = None
        gv, err = ask_genie(b["q"], space_id)
        if gv is not None and gt is not None and gt != 0:
            diff = abs(gv - gt) / abs(gt)
            st = "PASS" if diff < 0.05 else ("WARN" if diff < 0.15 else "FAIL")
        elif gv is None:
            st = f"FAIL {err or 'no answer'}"
        else:
            st = "PASS" if gv == gt else "FAIL"
        print(f"Q{i} {st} gt={gt} genie={gv}")
        results.append((i, b["q"][:60], float(gt) if gt is not None else None, float(gv) if gv is not None else None, st))
        time.sleep(1)
    passed = sum(1 for r in results if r[4] == "PASS")
    rate = passed / len(results) * 100 if results else 0
    print(f"Score: {passed}/{len(results)} ({rate:.0f}%)")
    return results, rate


benchmarks_v1 = [
    {
        "q": "How many distinct production lines had at least one defect_detected event in calendar year 2024?",
        "gt": f"SELECT CAST(COUNT(DISTINCT production_line_id) AS BIGINT) FROM {fqn}.production_events WHERE event_type = 'defect_detected' AND YEAR(CAST(event_date AS DATE)) = 2024",
    },
    {
        "q": "What is the sum of downtime_minutes from quality_metrics_daily for all rows in 2024 where the plant is in Michigan (join to plants by state)?",
        "gt": f"SELECT CAST(ROUND(SUM(q.downtime_minutes), 0) AS BIGINT) FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id = p.plant_id WHERE p.state = 'Michigan' AND YEAR(CAST(q.date AS DATE)) = 2024",
    },
    {
        "q": "What is the total units_produced summed from quality_metrics_daily for January through March 2024 (Q1) only?",
        "gt": f"SELECT CAST(SUM(q.units_produced) AS BIGINT) FROM {fqn}.quality_metrics_daily q WHERE CAST(q.date AS DATE) >= DATE '2024-01-01' AND CAST(q.date AS DATE) < DATE '2024-04-01'",
    },
    {
        "q": "How many production lines had 500 or more unit_produced events in calendar year 2024?",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM ( SELECT production_line_id FROM {fqn}.production_events WHERE event_type = 'unit_produced' AND YEAR(CAST(event_date AS DATE)) = 2024 GROUP BY production_line_id HAVING COUNT(*) >= 500 ) t",
    },
    {
        "q": "How many safety incidents are both severity Critical and category Equipment Malfunction?",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM {fqn}.safety_incidents WHERE severity = 'Critical' AND category = 'Equipment Malfunction'",
    },
    {
        "q": "How many operators work at plants located in Michigan (use operators.plant_id joined to plants)?",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM {fqn}.operators o JOIN {fqn}.plants p ON o.plant_id = p.plant_id WHERE p.state = 'Michigan'",
    },
    {
        "q": "For calendar year 2024 quality_metrics_daily only, what is the overall defect rate in percent rounded to a whole number: 100 times the sum of defects_found divided by the sum of units_produced?",
        "gt": f"SELECT CAST(ROUND(100.0 * SUM(defects_found) / SUM(units_produced), 0) AS BIGINT) FROM {fqn}.quality_metrics_daily WHERE YEAR(CAST(date AS DATE)) = 2024",
    },
    {
        "q": "How many production events in 2024 have event_type inspection_passed?",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM {fqn}.production_events WHERE event_type = 'inspection_passed' AND YEAR(CAST(event_date AS DATE)) = 2024",
    },
    {
        "q": "How many quality_metrics_daily rows in 2024 have oee_score strictly below 0.70?",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM {fqn}.quality_metrics_daily WHERE YEAR(CAST(date AS DATE)) = 2024 AND oee_score < 0.70",
    },
    {
        "q": "What is the sum of employee_count for all plants in Texas?",
        "gt": f"SELECT CAST(COALESCE(SUM(p.employee_count), 0) AS BIGINT) FROM {fqn}.plants p WHERE p.state = 'Texas'",
    },
    {
        "q": "How many equipment_feedback rows are for production lines at the EV Battery Innovation Center (join feedback to production_lines and match that plant)?",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM {fqn}.equipment_feedback f JOIN {fqn}.production_lines pl ON f.production_line_id = pl.line_id JOIN {fqn}.plants p ON pl.plant_id = p.plant_id WHERE p.plant_name = 'EV Battery Innovation Center'",
    },
    {
        "q": "For plant P003 only and calendar year 2024, what is average OEE as a whole percent rounded to an integer (average oee_score times 100, then round)?",
        "gt": f"SELECT CAST(ROUND(AVG(q.oee_score) * 100, 0) AS BIGINT) FROM {fqn}.quality_metrics_daily q WHERE q.plant_id = 'P003' AND YEAR(CAST(q.date AS DATE)) = 2024",
    },
]
print(len(benchmarks_v1), "benchmarks loaded (v2 harder suite)")


## Run A/B benchmark pass

Same **v2** questions as notebook **03**; 5% relative tolerance when ground truth is non-zero. Expect the **without-examples** space to slip **before** the primary space on harder joins and ratios.


In [ ]:
results_with, rate_with = run_benchmarks(
    benchmarks_v1, SID_WITH, "=== WITH in-space examples (+ benchmarks from 03) ==="
)
results_no, rate_no = run_benchmarks(
    benchmarks_v1, SID_NO, "=== WITHOUT in-space examples (instructions only) ==="
)

schema = StructType(
    [
        StructField("benchmark_id", IntegerType()),
        StructField("question", StringType()),
        StructField("ground_truth", DoubleType()),
        StructField("genie_answer", DoubleType()),
        StructField("status", StringType()),
    ]
)
from pyspark.sql import functions as F

df_w = spark.createDataFrame(results_with, schema).withColumn("variant", F.lit("with_examples"))
df_n = spark.createDataFrame(results_no, schema).withColumn("variant", F.lit("no_examples"))
display(df_w.unionByName(df_n))

print(f"Pass rate WITH examples: {rate_with:.1f}%")
print(f"Pass rate WITHOUT examples: {rate_no:.1f}%")


## Save run history

Appends to `genie_benchmark_runs` for notebook **09** (monitoring).


In [ ]:
ts = datetime.now().isoformat()
sch = StructType(
    [
        StructField("benchmark_id", IntegerType()),
        StructField("question", StringType()),
        StructField("ground_truth", DoubleType()),
        StructField("genie_answer", DoubleType()),
        StructField("status", StringType()),
        StructField("run_timestamp", StringType()),
        StructField("pass_rate", DoubleType()),
        StructField("run_label", StringType()),
    ]
)
rows = []
rows += [(r[0], r[1], r[2], r[3], r[4], ts, rate_with, "manufacturing_v1_with_examples") for r in results_with]
rows += [(r[0], r[1], r[2], r[3], r[4], ts, rate_no, "manufacturing_v1_no_examples") for r in results_no]

tbl = f"{fqn}.genie_benchmark_runs"
try:
    spark.createDataFrame(rows, sch).write.mode("append").saveAsTable(tbl)
except Exception:
    spark.sql(f"DROP TABLE IF EXISTS {tbl}")
    spark.createDataFrame(rows, sch).write.saveAsTable(tbl)
print(f"Saved to {tbl}")
